In [3]:
import pandas as pd
import os
import ast
import re

In [4]:
list_name = os.listdir('./')
study_group = 'EXAM'

In [5]:
list_name

['demo_script.ipynb',
 'EXAM_nhanes_codebook_ARX_F_resultados.csv',
 'EXAM_nhanes_codebook_AUX1_resultados.csv',
 'EXAM_nhanes_codebook_AUXAR_B_resultados.csv',
 'EXAM_nhanes_codebook_AUXAR_C_resultados.csv',
 'EXAM_nhanes_codebook_AUXAR_D_resultados.csv',
 'EXAM_nhanes_codebook_AUXAR_E_resultados.csv',
 'EXAM_nhanes_codebook_AUXAR_F_resultados.csv',
 'EXAM_nhanes_codebook_AUXAR_G_resultados.csv',
 'EXAM_nhanes_codebook_AUXAR_I_resultados.csv',
 'EXAM_nhanes_codebook_AUXAR_J_resultados.csv',
 'EXAM_nhanes_codebook_AUXAR_resultados.csv',
 'EXAM_nhanes_codebook_AUXTYM_B_resultados.csv',
 'EXAM_nhanes_codebook_AUXTYM_C_resultados.csv',
 'EXAM_nhanes_codebook_AUXTYM_D_resultados.csv',
 'EXAM_nhanes_codebook_AUXTYM_E_resultados.csv',
 'EXAM_nhanes_codebook_AUXTYM_F_resultados.csv',
 'EXAM_nhanes_codebook_AUXTYM_G_resultados.csv',
 'EXAM_nhanes_codebook_AUXTYM_I_resultados.csv',
 'EXAM_nhanes_codebook_AUXTYM_J_resultados.csv',
 'EXAM_nhanes_codebook_AUXTYM_resultados.csv',
 'EXAM_nhanes_code

In [6]:

name_tables = [palavra.split("EXAM_nhanes_codebook_")[1] for palavra in list_name if palavra.endswith('.csv')]
name_tables = [palavra.split("_") for palavra in name_tables if palavra.endswith('resultados.csv')]

nome_unique = []
for lista in name_tables:
    if len(lista[0])>1:
        nome_unique.append(lista[0])
    elif len(lista[0])==1:
        nome_unique.append(lista[1])
        
nome_unique = list(set(nome_unique))

In [7]:
def process_string_to_dict(string):
    string = string.split('; Count')[0]
    # Substituir os `;` por vírgulas e remover espaços extras
    formatted_string = re.sub(r";", ",", string)
    formatted_string = re.sub(r"=", ":", formatted_string)
    # Adicionar chaves externas para transformá-la em um dicionário
    formatted_string = "{" + formatted_string + "}"
    # Substituir chaves como "Code.or.Value" por "Code" e "Value.Description" por "Description"
    formatted_string = re.sub(r"Code\.or\.Value", '"Code"', formatted_string)
    formatted_string = re.sub(r"Value\.Description", '"Description"', formatted_string)

    return eval(formatted_string)

In [9]:
for nome in nome_unique:
    lista_filtrada = [item for item in list_name if nome in item]
    # print()
    # print("********************************")
    # print(lista_filtrada)
    
    df_now_total = pd.DataFrame()
    for item in lista_filtrada:
        df_now = pd.read_csv(item)
        df_now_total = pd.concat([df_now_total, df_now])
        
        
    list_Variable_Name = df_now_total['Variable_Name'].unique()
    df_codebook_final = pd.DataFrame()
    for variable in list_Variable_Name:
        df_now = df_now_total[df_now_total['Variable_Name']==variable]
        list_english = list(df_now['English_Text'].unique())
        list_Target = list(df_now['Target'].unique())
        list_English_Instructions = list(df_now['English_Instructions'].unique())
        df_now['English_Text'] = str(list_english)
        df_now['Target'] = str(list_Target)
        df_now['English_Instructions'] = str(list_English_Instructions)
        df_codebook_final = pd.concat([df_codebook_final, df_now])
    
    #display(df_codebook_final)

    rows = []
    for _, row in df_codebook_final.iterrows():
        # Ignorar linhas onde Tabela_Valores é NaN
        if pd.isna(row['Tabela_Valores']):
            rows.append({
                    "Column": row["Variable_Name"],
                    "Code": None,
                    "Label": None,
                    "English_Text": row["English_Text"],
                    "Target": row["Target"],
                    "English_Instructions": row["English_Instructions"]
                })
            continue
        
        # Extrair os dados de Code e Value.Description
        valores = row['Tabela_Valores']
        
        try:
            # Usando ast.literal_eval para transformar a string em dicionário
            parsed_values = process_string_to_dict(valores)
            codes = parsed_values["Code"]
            codes = [str(item).title() for item in codes]
            descriptions = parsed_values["Description"]
            descriptions = [str(item).title() for item in descriptions]
            # Criar uma nova linha para cada combinação de Code e Description
            for code, description in zip(codes, descriptions):
                rows.append({
                    "Column": row["Variable_Name"],
                    "Code": code,
                    "Label": description,
                    "English_Text": row["English_Text"],
                    "Target": row["Target"],
                    "English_Instructions": row["English_Instructions"]
                })
        except Exception as e:
            print(f"Erro ao processar a linha: {row['Tabela_Valores']}")
            print(e)

    # Criar um novo DataFrame com os dados processados
    df_expanded = pd.DataFrame(rows)
    df_expanded['Class'] = None
    df_expanded['Other For'] = None
    
    df_expanded.drop_duplicates(inplace=True)
    df_expanded.reset_index(inplace=True, drop=True)
    
    df_expanded.to_csv(f'./EXAM_SDD/{study_group}_{nome}_codebook.csv', sep=',', encoding='utf-8')
    
    dictionary_mapping = df_now_total[['Variable_Name']]
    dictionary_mapping = dictionary_mapping.drop_duplicates(subset='Variable_Name', keep='first')
    dictionary_mapping['Attribute'] = None
    dictionary_mapping['attributeOf'] = None
    dictionary_mapping['Unit'] = None
    dictionary_mapping['Time'] = None
    dictionary_mapping['Entity'] = None
    dictionary_mapping['Role'] = None
    dictionary_mapping['Relation'] = None
    dictionary_mapping['inRelationTo'] = None
    dictionary_mapping['wasDerivedFrom'] = None
    dictionary_mapping['wasGeneratedBy'] = None
    dictionary_mapping.reset_index(inplace=True, drop=True)
    dictionary_mapping.rename(columns={"Variable_Name": "Column"}, inplace=True)
    dictionary_mapping.reset_index(inplace=True, drop=True)
    dictionary_mapping.to_csv(f'./EXAM_SDD/{study_group}_{nome}_dictionary_mapping.csv', sep=',', encoding='utf-8')
    
    
    
    
    # display(df_expanded)
    # display(dictionary_mapping)
    # break

C:\Users\l-oen\AppData\Local\Temp\ipykernel_25392\1633740721.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['English_Text'] = str(list_english)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_25392\1633740721.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['Target'] = str(list_Target)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_25392\1633740721.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

Erro ao processar a linha: Code.or.Value=["1", "2", "3", "4", "5", "6", "."]; Value.Description=["Chest/abdomen surgery past three weeks", "Myocardial infarction (or"heart attack") in the past six weeks", "Told by dr had aneurysm in the brain or had a stroke", "Have severe neck or back pain today", "Difficult in bending or straightening right knee", "Had right knee or right hip replacement", "Missing"]; Count=["12", "8", "102", "38", "69", "64", "2000"]; Cumulative=["12", "20", "122", "160", "229", "293", "2293"]; Skip.to.Item=["NA", "NA", "NA", "NA", "NA", "NA", "NA"]
invalid syntax. Perhaps you forgot a comma? (<string>, line 1)
Erro ao processar a linha: Code.or.Value=["1", "2", "3", "4", "5", "6", "."]; Value.Description=["Chest/abdomen surgery past three weeks", "Myocardial infarction (or"heart attack") in the past six weeks", "Told by dr had aneurysm in the brain or had a stroke", "Have severe neck or back pain today", "Difficult in bending or straightening right knee", "Had righ

C:\Users\l-oen\AppData\Local\Temp\ipykernel_25392\1633740721.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['English_Text'] = str(list_english)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_25392\1633740721.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['Target'] = str(list_Target)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_25392\1633740721.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

Erro ao processar a linha: Code.or.Value=["0", "1", "2", "3", "4", "5", "6", "."]; Value.Description=["Valid data", "Jewelry or other objects not removed", "Non-removable objects (includes prostheses, implants)", "Excessive x-ray "noise" due to obesity", "Body parts out of scan region", "Positioning problem", "Other (includes panniculus, participant motion, unknown artifacts)", "Missing"]; Count=["6076", "0", "3", "3", "28", "3", "325", "1282"]; Cumulative=["6076", "6076", "6079", "6082", "6110", "6113", "6438", "7720"]; Skip.to.Item=["NA", "NA", "NA", "NA", "NA", "NA", "NA", "NA"]
invalid syntax. Perhaps you forgot a comma? (<string>, line 1)
Erro ao processar a linha: Code.or.Value=["0", "1", "2", "3", "4", "5", "6", "."]; Value.Description=["Valid data", "Jewelry or other objects not removed", "Non-removable objects (includes prostheses, implants)", "Excessive x-ray "noise" due to obesity", "Body parts out of scan region", "Positioning problem", "Other (includes panniculus, particip

C:\Users\l-oen\AppData\Local\Temp\ipykernel_25392\1633740721.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['English_Text'] = str(list_english)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_25392\1633740721.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['Target'] = str(list_Target)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_25392\1633740721.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

Erro ao processar a linha: Code.or.Value=["0", "1", "2", "3", "4", "5", "6", "."]; Value.Description=["Valid data", "Jewelry or other objects not removed", "Non-removable objects (includes prostheses, implants)", "Excessive x-ray "noise" due to obesity", "Body parts out of scan region", "Positioning problem", "Other (includes panniculus, participant motion, unknown artifacts)", "Missing"]; Count=["6076", "0", "3", "3", "28", "3", "325", "1282"]; Cumulative=["6076", "6076", "6079", "6082", "6110", "6113", "6438", "7720"]; Skip.to.Item=["NA", "NA", "NA", "NA", "NA", "NA", "NA", "NA"]
invalid syntax. Perhaps you forgot a comma? (<string>, line 1)
Erro ao processar a linha: Code.or.Value=["0", "1", "2", "3", "4", "5", "6", "."]; Value.Description=["Valid data", "Jewelry or other objects not removed", "Non-removable objects (includes prostheses, implants)", "Excessive x-ray "noise" due to obesity", "Body parts out of scan region", "Positioning problem", "Other (includes panniculus, particip

C:\Users\l-oen\AppData\Local\Temp\ipykernel_25392\1633740721.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['English_Text'] = str(list_english)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_25392\1633740721.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['Target'] = str(list_Target)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_25392\1633740721.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

Erro ao processar a linha: Code.or.Value=["1", "2", "4", "5", "."]; Value.Description=["Spine scan completed, all vertebrae are valid", "Spine scan completed, but one or more vertebrae are 
      invalid", "Spine not scanned, weight > 450 lbs", "Spine not scanned, other reason", "Missing"]; Count=["1274", "1199", "2", "423", "0"]; Cumulative=["1274", "2473", "2475", "2898", "2898"]; Skip.to.Item=["NA", "NA", "NA", "NA", "NA"]
unterminated string literal (detected at line 1) (<string>, line 1)
Erro ao processar a linha: Code.or.Value=["0", "1", "3", "4", "5", "6", "."]; Value.Description=["Valid data", "Removable or non-removable objects", "Excessive x-ray noise due to morbid obesity", "Insufficient scan area", "Movement", "Other (degenerative diseases, spinal fusion, 
    fractures)", "Missing"]; Count=["2219", "18", "0", "7", "3", "226", "425"]; Cumulative=["2219", "2237", "2237", "2244", "2247", "2473", "2898"]; Skip.to.Item=["NA", "NA", "NA", "NA", "NA", "NA", "NA"]
unterminated str

C:\Users\l-oen\AppData\Local\Temp\ipykernel_25392\1633740721.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['English_Text'] = str(list_english)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_25392\1633740721.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['Target'] = str(list_Target)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_25392\1633740721.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val